# Demo 3 — FA (Foundation Agents Protocol)

Source: [github.com/FoundationAgents/ai-link-net](https://github.com/FoundationAgents/ai-link-net)

**Scenario:** an equity analyst entity needs sector context. It **discovers** a macro analyst on the network — without being told its URL — sends a delegated question, and weaves the response into its brief.

```
                    ┌────────────────┐
                    │   CloudHost    │   relay (no entities of its own)
                    └────┬───────┬───┘
              parent     │       │     parent
                  ┌──────┘       └────────┐
            ┌─────┴────────┐         ┌────┴───────┐
            │  EquityHost  │         │  MacroHost │
            │              │         │            │
            │  alice ──┐   │         │  Macro     │
            │          ▼   │         │  Analyst   │
            │   Equity     │         │            │
            │   Analyst ───┼─────────┼──►  ◀──┐   │
            │              │         │        │   │
            │              │         │   reply│   │
            │   ◄──────────┼─────────┼────────┘   │
            └──────────────┘         └────────────┘

   1. alice → EquityAnalyst:      "Analyze NVDA, include sector context"
   2. EquityAnalyst (LLM loop) → fetches NVDA via stock MCP
   3. EquityAnalyst → host.get_discoverable_entities(include_parent=True)
                       finds MacroAnalyst on MacroHost
   4. EquityAnalyst → MacroAnalyst (cross-host INVOKE through CloudHost)
   5. MacroAnalyst (LLM loop) → returns sector outlook
   6. EquityAnalyst → weaves both into a brief → alice
```

A2A would require alice (or the equity analyst) to know each callee's URL up front. Here the network has a directory of public entities, and `host.get_discoverable_entities(include_parent=True)` returns cards across hosts.

## First-time setup

Run this once in a terminal, from the directory where you want the repo:

```bash
git clone https://github.com/jackwu502/ivado-protocol.git
cd ivado-protocol

python3.12 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name ivado-lab --display-name "ivado-lab (3.12)"
```

Create your local `.env` file:

```bash
cp .env.example .env
```

Then edit `.env` and fill in one credential route:

```bash
# Option 1: Anthropic direct
ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_MODEL=claude-sonnet-4-6

# Option 2: OpenRouter-compatible Anthropic endpoint
# ANTHROPIC_BASE_URL=https://openrouter.ai/api
# ANTHROPIC_API_KEY=sk-or-v1-...
# ANTHROPIC_MODEL=anthropic/claude-sonnet-4.5
```

Do not commit `.env`; it is intentionally gitignored.

Start Jupyter from the repo root and select the `ivado-lab (3.12)` kernel:

```bash
python -m jupyter lab
```


Demo 3 also needs the FA reference implementation (`ai-link-net`). The install cell below tries these options in order:

1. Use `fp` / `aln` if they are already installed.
2. Use `ALN_LOCAL_PATH` from `.env`, if set.
3. Use a local clone at `./ai-link-net` or `../ai-link-net`.
4. Install from `https://github.com/FoundationAgents/ai-link-net`.

If GitHub access is private for you, clone `ai-link-net` next to this repo or set `ALN_LOCAL_PATH` in `.env`.

In [ ]:
# ── Bootstrap: resolve paths to shared/ and sibling helpers ──
import sys
from pathlib import Path
HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))   # so `from shared.X import Y` works
sys.path.insert(0, str(HERE))   # so sibling helpers import directly
STOCK_MCP_SERVER = str(ROOT / "shared" / "stock_mcp_server.py")


In [ ]:
assert sys.version_info >= (3, 12), "ai-link-net needs Python 3.12+"

# Dependencies for the wrapped MCP server and the analyst loop
%pip install -q mcp anthropic yfinance python-dotenv

import importlib.util
import os
import subprocess

def _have_ai_link_net():
    return importlib.util.find_spec("fp") is not None and importlib.util.find_spec("aln") is not None

def _pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if _have_ai_link_net():
    print("ai-link-net already importable")
else:
    candidates = []
    if os.getenv("ALN_LOCAL_PATH"):
        candidates.append(Path(os.environ["ALN_LOCAL_PATH"]).expanduser())
    candidates.append(ROOT / "ai-link-net")
    candidates.append(ROOT.parent / "ai-link-net")

    local_clone = next((p for p in candidates if p and p.exists()), None)
    if local_clone:
        print(f"installing ai-link-net from local clone: {local_clone}")
        _pip_install("-e", str(local_clone))
    else:
        print("installing ai-link-net from GitHub")
        _pip_install('git+https://github.com/FoundationAgents/ai-link-net.git')

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("Python:", sys.version.split()[0])

## Setup

`fp/` is the protocol layer; `aln/` provides `MCPHandler`. Patch `fp/Host` so `TOOL` entities resolve to `MCPHandler`. Silence logs and skip disk persistence for the demo.

In [ ]:
import asyncio, json
from unittest.mock import patch
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="WARNING", format="<level>{level}</level> | {message}")
patch("fp.host.Host.save").start()

from fp import Host
from aln.app.handlers import create_entity_handler

_orig = Host._resolve_entity_handler
def _resolve(self, entity, handler, provider, system_prompt, handler_config):
    return _orig(self, entity, handler, provider, system_prompt, handler_config) \
        or create_entity_handler(entity=entity, kind=entity.kind, provider=provider,
                                 system_prompt=system_prompt, handler_config=handler_config)
Host._resolve_entity_handler = _resolve
print("ready")

## Set up the network

Two hosts under a relay. The equity analyst lives with alice on `EquityHost`; the macro analyst lives on `MacroHost`. Alice does not know macro exists yet — equity will discover it at query time.

In [ ]:
from fp import Host, Message, MessageKind
from fp.core.base import EntityKind
from fp.message import FriendRequestPayload
from fa_analysts import make_equity_analyst, make_macro_analyst, FRIEND_KINDS

cloud = Host(name="CloudHost")
host_a = Host(name="EquityHost", port=18101)
host_b = Host(name="MacroHost",  port=18102)
host_a.set_parent_host(cloud)
host_b.set_parent_host(cloud)

# Macro analyst on host_b (will be discovered, not pre-configured)
macro = make_macro_analyst(host_b)

# Equity analyst on host_a — looks up macro at query time
equity = make_equity_analyst(host_a)

# Alice (the user) on host_a
alice_inbox: asyncio.Queue[Message] = asyncio.Queue()
async def alice_handler(msg):
    if msg.kind not in FRIEND_KINDS:
        await alice_inbox.put(msg)
alice = host_a.register_entity("alice", kind=EntityKind.HUMAN, handler=alice_handler)

print(f"{alice.name:<14}  {alice.address.address}")
print(f"{equity.name:<14}  {equity.address.address}")
print(f"{macro.name:<14}  {macro.address.address}")

## Discovery

The equity analyst's host walks the federation tree and lists every public entity it can see — including the macro analyst on the other host. **No URL was pre-configured.**

In [ ]:
for card in host_a.get_discoverable_entities(include_parent=True):
    print(f"{card.name:<14}  {card.address.address}  (kind={card.kind})")

## Friend handshake (alice ↔ equity)

The equity ↔ macro handshake happens lazily on the first delegation call inside `make_equity_analyst`.

In [ ]:
await alice.send_message(
    to=equity.entity_card,
    message=Message(kind=MessageKind.FRIEND_REQUEST,
                    payload=FriendRequestPayload(sender_card=alice.entity_card)),
)
await asyncio.sleep(0.5)
print("alice friends:", list(alice.friends.keys()))

## Alice asks one question

The equity analyst plans, fetches NVDA data, decides sector context would help, and **delegates to the macro analyst it discovered** — across hosts, through the relay. The macro analyst replies; the equity analyst combines and returns its brief.

In [ ]:
await alice.send_message(
    to=equity.entity_card,
    message=Message(
        kind=MessageKind.INVOKE,
        payload={"text": "Analyze NVDA briefly. Include sector context."},
    ),
)
reply = await asyncio.wait_for(alice_inbox.get(), timeout=300)
print(reply.payload["text"])

## CLI entry point

The non-coding workflow uses the `aln` CLI and a WebUI (`aln init`, `aln ui`, `bash demo/quickstart.sh`). Below is just the help text.

In [ ]:
import shutil, subprocess
from pathlib import Path
aln = shutil.which("aln") or str(Path(sys.executable).parent / "aln")
if Path(aln).exists():
    print(subprocess.run([aln, "-h"], capture_output=True, text=True).stdout[:1200])
else:
    print("`aln` not on PATH (does not affect the demo above)")